#  Langfuse Evaluation을 사용한 RAG 답변 평가

- Langfuse를 활용한 RAG 시스템 평가 방법을 이해한다
- 데이터셋 생성 및 Langfuse 업로드 방법을 습득한다
- Experiment Runner SDK (v3)를 사용한 평가 자동화를 학습한다
- 아이템 레벨 및 런 레벨 Evaluator를 구현한다

---


## 환경 설정 및 준비

`(1) Env 환경변수`

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [2]:
import os
from glob import glob

from pprint import pprint
import json

import uuid

import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

`(3) langfuse handler 설정`

In [3]:
from langfuse.langchain import CallbackHandler

# LangChain 콜백 핸들러 생성
langfuse_handler = CallbackHandler()

`(4) Test Data`

In [4]:
# Test 데이터셋에 대한 QA 생성 결과를 리뷰한 후 다시 로드
import pandas as pd
df_qa_test = pd.read_excel("data/testset.xlsx")

print(f"테스트셋: {df_qa_test.shape[0]}개 문서")
df_qa_test.head(2)

테스트셋: 49개 문서


,user_input,reference_contexts,reference,synthesizer_name
0,"Tesla, Inc.는 미국에서 어떤 역할을 하고 있으며, 이 회사의 주요 제품과 ...","['Tesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사입니다. 이 회...","Tesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사로, 전기 자동차(...",single_hop_specifc_query_synthesizer
1,Forbes Global 2000에서 테슬라 순위 뭐야?,['Tesla의 차량 생산은 2008년 Roadster로 시작하여 Model S (...,테슬라는 Forbes Global 2000에서 69위에 랭크되었습니다.,single_hop_specifc_query_synthesizer


---

## **검색 도구 정의** 

### 1) **벡터스토어** 로드

- **Chroma DB** 설정에서 모델, 컬렉션명, 저장 경로 지정

In [5]:
# 벡터 저장소 로드
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

chroma_db = Chroma(
    collection_name="db_korean_cosine_metadata",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
)

print(chroma_db._collection.count())

39


In [6]:
# 벡터저장소 검색기 생성
chroma_k = chroma_db.as_retriever(
    search_kwargs={'k': 4},
)

# 벡터저장소 검색기를 사용하여 검색
query = "테슬라의 회장은 누구인가요?"

retrieved_docs = chroma_k.invoke(query)

# 검색 결과 출력
for doc in retrieved_docs:
    print(f"- {doc.page_content} [출처: {doc.metadata['source']}]")
    print("-"*200)
    print()

- [출처] 이 문서는 테슬라(Tesla)에 대한 문서입니다.
----------------------------------
### Roadster (2005–2009)

Elon Musk는 주류 차량으로 확장하기 전에 프리미엄 스포츠카로 시작하는 전략에 초점을 맞춰 적극적인 역할을 수행했습니다. 후속 자금 조달에는 Valor Equity Partners (2006)와 Sergey Brin, Larry Page, Jeff Skoll과 같은 기업가의 투자가 포함되었습니다.

2007년 8월, Eberhard는 CEO에서 물러나라는 요청을 받았고, Tarpenning은 2008년 1월에 이어졌습니다. Michael Marks는 Ze'ev Drori가 인수하기 전에 임시 CEO를 역임했으며, Musk는 2008년 10월에 인수했습니다. Eberhard는 2009년 6월 Musk를 상대로 소송을 제기했지만 나중에 기각되었습니다. [출처: data\Tesla_KR.md]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

- [출처] 이 문서는 테슬라(Tesla)에 대한 문서입니다.
----------------------------------
Tesla는 내부 고발자 보복, 근로자 권리 침해, 안전 결함, 홍보 부족, Musk의 논란의 여지가 있는 발언과 관련된 소송, 정부 조사 및 비판에 직면했습니다.

## 역사

### 창립 (2003–2004)

Tesla Motors, Inc.는 2003년 7월 1일에 Martin Eberhard와 Marc Tarpenning에 의해 설립되었으며, 각각 CEO와 CFO를 역임했습니다. Ian Wright는 얼마 지나지 않

### 2) **BM25 검색기** 준비

- **BM25 검색기** 구현으로 문서 유사도 기반 검색 가능

- **한국어 텍스트 처리**를 위한 **Kiwi 토크나이저** 설정

- 참고: https://github.com/bab2min/kiwipiepy

In [7]:
# korean_docs 파일을 로드 (jsonlines 파일)
def load_jsonlines(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        docs = [json.loads(line) for line in f]
    return docs

korean_docs = load_jsonlines('korean_docs_final.jsonl')
print(f"로드된 문서: {len(korean_docs)}개")
pprint(korean_docs[0])

로드된 문서: 39개
('{"id":null,"metadata":{"source":"data/테슬라_KR.md","company":"테슬라","language":"ko"},"page_content":"<Document>\\nTesla, '
 'Inc.는 미국의 다국적 자동차 및 청정 에너지 회사입니다. 이 회사는 전기 자동차(BEV), 고정형 배터리 에너지 저장 장치, 태양 '
 '전지판, 태양광 지붕널 및 관련 제품/서비스를 설계, 제조 및 판매합니다. 2003년 7월 Martin Eberhard와 Marc '
 'Tarpenning이 Tesla Motors로 설립했으며, Nikola Tesla를 기리기 위해 명명되었습니다. Elon Musk는 '
 '2004년 Tesla의 초기 자금 조달을 주도하여 2008년에 회장 겸 CEO가 '
 "되었습니다.\\n</Document>\\n<Source>이 문서는 미국 전기차 회사인 '테슬라'에 대한 "
 '문서입니다.</Source>","type":"Document"}')


In [8]:
from langchain_core.documents import Document  # Document 클래스 임포트

# 문자열 리스트를 Document 객체로 변환
if isinstance(korean_docs[0], str):  # 첫 번째 항목이 문자열인지 확인
    documents = [
        Document(
            page_content=json.loads(data)['page_content'],  # 문자열을 파이썬 객체로 변환
            metadata=json.loads(data)['metadata']
        ) 
        for i, data in enumerate(korean_docs)
    ]
else:
    documents = korean_docs

print(f"변환된 문서: {len(documents)}개")
pprint(documents[0])

변환된 문서: 39개
Document(metadata={'source': 'data/테슬라_KR.md', 'company': '테슬라', 'language': 'ko'}, page_content="<Document>\nTesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사입니다. 이 회사는 전기 자동차(BEV), 고정형 배터리 에너지 저장 장치, 태양 전지판, 태양광 지붕널 및 관련 제품/서비스를 설계, 제조 및 판매합니다. 2003년 7월 Martin Eberhard와 Marc Tarpenning이 Tesla Motors로 설립했으며, Nikola Tesla를 기리기 위해 명명되었습니다. Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도하여 2008년에 회장 겸 CEO가 되었습니다.\n</Document>\n<Source>이 문서는 미국 전기차 회사인 '테슬라'에 대한 문서입니다.</Source>")


In [9]:
# BM25 검색기를 사용하기 위한 준비
from langchain_community.retrievers import BM25Retriever
from ranx_k.tokenizers import KiwiTokenizer

kiwi_tokenizer = KiwiTokenizer(
    use_stopwords=False,  # 불용어 사용 안 함
    pos_filter=[]         # 품사 필터 없음
)

# Kiwi 토크나이저를 사용하는 전처리 함수
def kiwi_preprocess(text: str) -> list[str]:
    return kiwi_tokenizer.tokenize(text)  # 문자열 리스트 반환

# BM25 검색기 생성 (한국어 토크나이저 적용)
bm25_db = BM25Retriever.from_documents(
    documents,
    preprocess_func=kiwi_preprocess,
    k=4,
)


In [10]:
# BM25 검색기를 사용하여 문서 검색
query = "테슬라의 회장은 누구인가요?"
retrieved_docs = bm25_db.invoke(query)

# 검색 결과 출력 
for i, doc in enumerate(retrieved_docs, 1):
    print(f"[검색 결과 {i}]")
    print(f"{doc.page_content}\n[출처: {doc.metadata['source']}]")
    print("-"*100)


[검색 결과 1]
<Document>
Tesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사입니다. 이 회사는 전기 자동차(BEV), 고정형 배터리 에너지 저장 장치, 태양 전지판, 태양광 지붕널 및 관련 제품/서비스를 설계, 제조 및 판매합니다. 2003년 7월 Martin Eberhard와 Marc Tarpenning이 Tesla Motors로 설립했으며, Nikola Tesla를 기리기 위해 명명되었습니다. Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도하여 2008년에 회장 겸 CEO가 되었습니다.
</Document>
<Source>이 문서는 미국 전기차 회사인 '테슬라'에 대한 문서입니다.</Source>
[출처: data/테슬라_KR.md]
----------------------------------------------------------------------------------------------------
[검색 결과 2]
<Document>
Tesla는 내부 고발자 보복, 근로자 권리 침해, 안전 결함, 홍보 부족, Musk의 논란의 여지가 있는 발언과 관련된 소송, 정부 조사 및 비판에 직면했습니다.

## 역사

### 창립 (2003–2004)

Tesla Motors, Inc.는 2003년 7월 1일에 Martin Eberhard와 Marc Tarpenning에 의해 설립되었으며, 각각 CEO와 CFO를 역임했습니다. Ian Wright는 얼마 지나지 않아 합류했습니다. 2004년 2월, Elon Musk는 750만 달러의 시리즈 A 자금 조달을 주도하여 회장 겸 최대 주주가 되었습니다. J. B. Straubel은 2004년 5월 CTO로 합류했습니다. 다섯 명 모두 공동 설립자로 인정받고 있습니다.

### Roadster (2005–2009)
</Document>
<Source>이 문서는 미국 전기차 회사인 '테슬라'에 대한 문서입니다.</Source>
[출처: data/테슬라_KR.md]
-

### 3) **Emsemble Hybrid Search** 준비

- **BM25**, **벡터 검색** 결과를 **rank-fusion** 알고리즘으로 통합 (**EnsembleRetriever**)

- 각 검색기의 **순위 점수**를 고려한 최종 순위 결정

- **중복 문서** 제거와 **재순위화** 자동 수행

- 두 검색 방식의 **장점을 결합**해 검색 품질 향상

In [11]:
from langchain_classic.retrievers import EnsembleRetriever

# 검색기 초기화 
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_db, chroma_k],
    weights=[0.5, 0.5],
)


In [12]:
query = "테슬라의 회장은 누구인가요?"
retrieved_docs = hybrid_retriever.invoke(query)

# 검색 결과 출력 
for doc in retrieved_docs:
    print(f"\n{doc.page_content}\n[출처: {doc.metadata['source']}]")
    print("-"*200)


<Document>
Tesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사입니다. 이 회사는 전기 자동차(BEV), 고정형 배터리 에너지 저장 장치, 태양 전지판, 태양광 지붕널 및 관련 제품/서비스를 설계, 제조 및 판매합니다. 2003년 7월 Martin Eberhard와 Marc Tarpenning이 Tesla Motors로 설립했으며, Nikola Tesla를 기리기 위해 명명되었습니다. Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도하여 2008년에 회장 겸 CEO가 되었습니다.
</Document>
<Source>이 문서는 미국 전기차 회사인 '테슬라'에 대한 문서입니다.</Source>
[출처: data/테슬라_KR.md]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

[출처] 이 문서는 테슬라(Tesla)에 대한 문서입니다.
----------------------------------
### Roadster (2005–2009)

Elon Musk는 주류 차량으로 확장하기 전에 프리미엄 스포츠카로 시작하는 전략에 초점을 맞춰 적극적인 역할을 수행했습니다. 후속 자금 조달에는 Valor Equity Partners (2006)와 Sergey Brin, Larry Page, Jeff Skoll과 같은 기업가의 투자가 포함되었습니다.

2007년 8월, Eberhard는 CEO에서 물러나라는 요청을 받았고, Tarpenning은 2008년 1월에 이어졌습니다. Michael Marks는 Ze'ev Drori가 인수하기 전에 임시 CEO를 역임했으며, Musk는 2008년 10월에 인수했습니다. Eberhard는

## **RAG Chain** 정의

- OpenAI gpt-4.1-mini 모델 활용

In [13]:
# 각 쿼리에 대한 검색 결과를 한꺼번에 Context로 전달해서 답변을 생성
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

def create_rag_chain(retriever, llm):

    template = """Answer the following question based on this context. If the context is not relevant to the question, just answer with '답변에 필요한 근거를 찾지 못했습니다.'

    [Context]
    {context}

    [Question]
    {question}

    [Answer]
    """

    prompt = ChatPromptTemplate.from_template(template)

    def format_docs(docs):
        return "\n\n".join([f"{doc.page_content}" for doc in docs])

    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()} 
        | prompt
        | llm
        | StrOutputParser()
    )

    return rag_chain

In [14]:
# RAG 체인 생성 및 테스트
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.5)

openai_rag_chain = create_rag_chain(hybrid_retriever, llm)

question = "테슬라의 회장은 누구인가요?"
answer = openai_rag_chain.invoke(question)

print(f"쿼리: {question}")
print(f"답변: {answer}") 

쿼리: 테슬라의 회장은 누구인가요?
답변: Elon Musk는 2008년에 Tesla의 회장 겸 CEO가 되었습니다. 따라서 테슬라의 회장은 Elon Musk입니다.


---

## **Langfuse를 활용한 평가**

[Langfuse](https://langfuse.com)는 LLM 애플리케이션의 **관찰성(Observability)** 과 **평가(Evaluation)** 를 위한 오픈소스 플랫폼입니다.

### 핵심 기능

| 기능 | 설명 |
|------|------|
| **Tracing** | LLM 호출, 체인, 에이전트 실행 추적 |
| **Dataset** | 평가용 테스트 데이터 관리 |
| **Experiment** | 체계적인 A/B 테스트 및 평가 실행 |
| **Evaluation** | 자동화된 품질 평가 |
| **Dashboard** | 실시간 모니터링 및 분석 |

### Experiment Runner SDK (v3)

```python
# 기본 사용법
result = dataset.run_experiment(
    name="experiment_name",
    task=task_function,           # 실행할 함수
    evaluators=[...],             # 아이템별 평가자
    run_evaluators=[...],         # 전체 집계 평가자
    max_concurrency=3,            # 동시 실행 수
)
```

### Evaluator 유형

| 유형 | 실행 시점 | 입력 | 용도 |
|------|----------|------|------|
| **Item Evaluator** | 각 아이템 실행 후 | `input, output, expected_output` | 개별 평가 |
| **Run Evaluator** | 전체 실험 완료 후 | `item_results` | 집계/요약 |

### Evaluation 반환 형식

```python
from langfuse import Evaluation

Evaluation(
    name="metric_name",    # 평가 지표 이름
    value=0.85,            # 점수 (숫자)
    comment="설명..."      # 선택적 코멘트
)
```

### 장점

1. **동시 실행**: `max_concurrency`로 병렬 처리
2. **자동 추적**: 모든 실행에 대한 자동 트레이싱
3. **에러 격리**: 개별 실패가 전체 실험을 중단시키지 않음
4. **유연한 평가**: 동기/비동기 평가자 모두 지원
5. **시각화**: 대시보드에서 결과 비교 및 분석


---
### 1) **Langfuse 환경 설정** 

In [15]:
from langfuse.langchain import CallbackHandler

# Langfuse 콜백 핸들러 생성
langfuse_handler = CallbackHandler()

In [16]:
from langfuse import get_client

# Langfuse 클라이언트 초기화
langfuse_client = get_client()

---
### 2) **평가용 데이터셋 업로드** 

In [17]:
df_qa_test.head(2)

,user_input,reference_contexts,reference,synthesizer_name
0,"Tesla, Inc.는 미국에서 어떤 역할을 하고 있으며, 이 회사의 주요 제품과 ...","['Tesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사입니다. 이 회...","Tesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사로, 전기 자동차(...",single_hop_specifc_query_synthesizer
1,Forbes Global 2000에서 테슬라 순위 뭐야?,['Tesla의 차량 생산은 2008년 Roadster로 시작하여 Model S (...,테슬라는 Forbes Global 2000에서 69위에 랭크되었습니다.,single_hop_specifc_query_synthesizer


In [18]:
# Langfuse에서 데이터셋 생성
name = "RAG_Evaluation_Dataset_Test"

try:
    dataset = langfuse_client.create_dataset(name=name)
    print(f"새 데이터셋 생성: {dataset.name}")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"기존 데이터셋 사용: {name}")
        dataset = langfuse_client.get_dataset(name=name)
    else:
        raise e

새 데이터셋 생성: RAG_Evaluation_Dataset_Test


In [19]:
# 평가용 데이터셋을 변환 
data=[
    {
        "user_input": row["user_input"],
        "reference": row["reference"],
        "reference_contexts": row["reference_contexts"],
    } for _, row in df_qa_test.iterrows()
]

print(f"평가용 데이터셋 아이템 수: {len(data)}개")

평가용 데이터셋 아이템 수: 49개


In [20]:
# 데이터셋 아이템 추가
for item in data:
    langfuse_client.create_dataset_item(
        dataset_name=name,
        input=item.get("user_input", ""),
        expected_output=item.get("reference", ""),
        metadata={
            "reference_contexts": item.get("reference_contexts", ""),            
        }
    )


# langfuse에 플러시 (저장)
langfuse_client.flush()

In [21]:
# 추가된 데이터셋 확인 (가져오기)
dataset = langfuse_client.get_dataset(name="RAG_Evaluation_Dataset_Test")
print(f"생성된 데이터셋: {dataset.name}")
print(f"데이터셋 아이템 수: {len(dataset.items)}개")

생성된 데이터셋: RAG_Evaluation_Dataset_Test
데이터셋 아이템 수: 49개


In [22]:
# 평가용 데이터셋 아이템 출력
for item in dataset.items[:5]:  # 처음 5개 아이템만 출력
    print(f"입력: {item.input}")
    print(f"기대 출력: {item.expected_output}")
    print(f"메타데이터: {item.metadata}")
    print("-"*200)

입력: Elon Musk의 560억 달러 급여 패키지가 델라웨어 법원에서 거부된 이유와 그의 Tesla에서의 초기 전략적 역할은 무엇인가요?
기대 출력: Elon Musk의 560억 달러 급여 패키지는 2024년 12월 델라웨어 법원에서 부적절한 이사회 승인으로 인해 거부되었습니다. Tesla에서의 초기 전략적 역할로는 주류 차량으로 확장하기 전에 프리미엄 스포츠카로 시작하는 전략에 초점을 맞추는 것이었습니다. 그는 Valor Equity Partners와 같은 투자자들로부터 자금을 조달하며 Tesla의 성장을 이끌었습니다.
메타데이터: {'reference_contexts': '[\'<1-hop>\\n\\n2024년 12월, 델라웨어 법원은 부적절한 이사회 승인을 이유로 Elon Musk의 560억 달러 급여 패키지를 거부했습니다.\\n\\n## 자동차 제품 및 서비스\\n\\n2024년 11월 현재 Tesla는 Model S, Model X, Model 3, Model Y, Semi 및 Cybertruck의 6가지 차량 모델을 제공합니다. 다음은 Tesla 모델 목록입니다.\', "<2-hop>\\n\\nElon Musk는 주류 차량으로 확장하기 전에 프리미엄 스포츠카로 시작하는 전략에 초점을 맞춰 적극적인 역할을 수행했습니다. 후속 자금 조달에는 Valor Equity Partners (2006)와 Sergey Brin, Larry Page, Jeff Skoll과 같은 기업가의 투자가 포함되었습니다.\\n\\n2007년 8월, Eberhard는 CEO에서 물러나라는 요청을 받았고, Tarpenning은 2008년 1월에 이어졌습니다. Michael Marks는 Ze\'ev Drori가 인수하기 전에 임시 CEO를 역임했으며, Musk는 2008년 10월에 인수했습니다. Eberhard는 2009년 6월 Musk를 상대로 소송을 제기했지만 나중에 기각되었습니다."]'}
-------------------------------------------------

---

## **평가 지표** (Evaluation Metrics)

RAG 시스템의 성능을 정량적으로 측정하기 위한 다양한 평가 지표입니다.

### 1) 검색(Retrieval) 평가 지표

| 지표 | 설명 | 범위 |
|------|------|------|
| **Hit Rate@K** | 상위 K개 문서에 정답 포함 비율 | 0~1 |
| **Precision@K** | 검색된 K개 중 관련 문서 비율 | 0~1 |
| **Recall@K** | 전체 관련 문서 중 검색된 비율 | 0~1 |
| **MRR** | 첫 번째 관련 문서 순위 역수 평균 | 0~1 |
| **NDCG@K** | 순위를 고려한 검색 품질 | 0~1 |

### 2) 생성(Generation) 평가 지표

#### 전통적 지표 (Reference-based)

| 지표 | 설명 | 용도 |
|------|------|------|
| **ROUGE-L** | 최장 공통 부분수열 기반 | 요약 평가 |
| **BLEU** | n-gram 정밀도 기반 | 번역 평가 |
| **BertScore** | 임베딩 기반 의미 유사도 | 의미 평가 |

#### LLM-as-Judge 지표

| 지표 | 설명 | 참조 필요 |
|------|------|----------|
| **Faithfulness** | 컨텍스트에 충실한지 (환각 방지) | Context |
| **Answer Relevancy** | 질문에 적절히 대응하는지 | ❌ |
| **Correctness** | 사실적 정확성 | Reference |
| **Groundedness** | 컨텍스트에 근거하는지 | Context |

### 3) Langfuse에서의 평가 구현

```python
from langfuse import Evaluation

# 아이템 레벨 평가자 예시
def my_evaluator(*, input, output, expected_output, **kwargs):
    score = calculate_score(output, expected_output)
    return Evaluation(
        name="my_metric",
        value=score,
        comment=f"점수: {score:.2f}"
    )
```

**참고**: [Langfuse Evaluation Docs](https://langfuse.com/docs/scores/model-based-evals)

---
### 3) **데이터셋 기반 평가 실행**


In [23]:
from rouge_score import rouge_scorer
from ranx_k.tokenizers import KiwiTokenizer
from openevals.llm import create_llm_as_judge
from openevals.prompts import CONCISENESS_PROMPT

# Kiwi 토크나이저 생성 (tokenize()가 이미 문자열 리스트 반환)
kiwi_tokenizer = KiwiTokenizer(use_stopwords=False, pos_filter=[])

# ROUGE 스코어 계산
rouge_scorer_instance = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    tokenizer=kiwi_tokenizer      # tokenize 메소드를 갖는 토크나이저 사용
)

# 간결성 평가자 (OpenEvals 사용)
conciseness_evaluator = create_llm_as_judge(
    prompt=CONCISENESS_PROMPT,
    feedback_key="conciseness",
    model="openai:gpt-4.1-mini",
)

In [24]:
from langfuse import get_client, Evaluation

# Langfuse 클라이언트 초기화
langfuse_client = get_client()

# 1. Task 함수 정의 - RAG 체인 실행
def rag_task(*, item, **kwargs):
    """RAG 체인을 실행하여 답변 생성"""
    question = item.input
    answer = openai_rag_chain.invoke(question)
    return answer

# 2. 아이템 레벨 Evaluator 정의

def rouge_evaluator(*, input, output, expected_output, **kwargs):
    """ROUGE 점수 평가"""
    try:
        rouge_results = rouge_scorer_instance.score(
            str(expected_output), 
            str(output)
        )
        rouge_l_score = rouge_results['rougeL'].fmeasure
        return Evaluation(
            name="rouge_l",
            value=rouge_l_score,
            comment=f"ROUGE-L F1: {rouge_l_score:.4f}"
        )
    except Exception as e:
        return Evaluation(name="rouge_l", value=0.0, comment=str(e))

def conciseness_llm_evaluator(*, input, output, expected_output, **kwargs):
    """OpenEvals 간결성 평가"""
    try:
        result = conciseness_evaluator(
            inputs=str(input),
            outputs=str(output),
        )
        score = 1.0 if result.get('score') else 0.0
        return Evaluation(
            name="conciseness",
            value=score,
            comment=result.get('comment', '')
        )
    except Exception as e:
        return Evaluation(name="conciseness", value=0.0, comment=str(e))

def length_evaluator(*, input, output, **kwargs):
    """응답 길이 평가"""
    length = len(str(output))
    return Evaluation(
        name="response_length",
        value=length,
        comment=f"응답 길이: {length}자"
    )

# 3. Run-level Evaluator 정의 (전체 실험 집계)

def average_rouge_evaluator(*, item_results, **kwargs):
    """전체 ROUGE 평균 계산"""
    scores = [
        eval.value for result in item_results
        for eval in result.evaluations
        if eval.name == "rouge_l" and eval.value is not None
    ]
    
    if not scores:
        return Evaluation(name="avg_rouge_l", value=None)
    
    avg = sum(scores) / len(scores)
    return Evaluation(
        name="avg_rouge_l",
        value=avg,
        comment=f"평균 ROUGE-L: {avg:.4f} ({len(scores)}개 항목)"
    )

def average_conciseness_evaluator(*, item_results, **kwargs):
    """전체 간결성 평균 계산"""
    scores = [
        eval.value for result in item_results
        for eval in result.evaluations
        if eval.name == "conciseness" and eval.value is not None
    ]
    
    if not scores:
        return Evaluation(name="avg_conciseness", value=None)
    
    avg = sum(scores) / len(scores)
    return Evaluation(
        name="avg_conciseness",
        value=avg,
        comment=f"평균 간결성: {avg:.2f} ({len(scores)}개 항목)"
    )

In [25]:
# Experiment Runner SDK를 사용한 평가 실행
dataset = langfuse_client.get_dataset("RAG_Evaluation_Dataset_Test")

# 실험 실행
result = dataset.run_experiment(
    name="RAG_Evaluation_Experiment",
    description="ROUGE와 간결성을 활용한 RAG 시스템 평가",
    task=rag_task,
    evaluators=[
        rouge_evaluator,
        conciseness_llm_evaluator,
        length_evaluator
    ],
    run_evaluators=[
        average_rouge_evaluator,
        average_conciseness_evaluator
    ],
    max_concurrency=3,  # 동시 실행 수 제한
)

# 결과 출력
print(result.format())

Propagated attribute 'experiment_item_metadata' value is over 200 characters (1775 chars). Dropping value.
Propagated attribute 'experiment_item_metadata' value is over 200 characters (1775 chars). Dropping value.
Propagated attribute 'experiment_item_metadata' value is over 200 characters (1775 chars). Dropping value.
Propagated attribute 'experiment_item_metadata' value is over 200 characters (1775 chars). Dropping value.
Propagated attribute 'experiment_item_metadata' value is over 200 characters (1984 chars). Dropping value.
Propagated attribute 'experiment_item_metadata' value is over 200 characters (1984 chars). Dropping value.
Propagated attribute 'experiment_item_metadata' value is over 200 characters (1984 chars). Dropping value.
Propagated attribute 'experiment_item_metadata' value is over 200 characters (1984 chars). Dropping value.
Propagated attribute 'experiment_item_metadata' value is over 200 characters (2057 chars). Dropping value.
Propagated attribute 'experiment_item

Individual Results: Hidden (49 items)\n💡 Set include_item_results=True to view them\n\n──────────────────────────────────────────────────\n🧪 Experiment: RAG_Evaluation_Experiment
📋 Run name: RAG_Evaluation_Experiment - 2026-03-14T01:40:17.555873Z - ROUGE와 간결성을 활용한 RAG 시스템 평가\n49 items\nEvaluations:\n  • rouge_l\n  • conciseness\n  • response_length\n\nAverage Scores:\n  • rouge_l: 0.566\n  • conciseness: 0.490\n  • response_length: 261.755\n\nRun Evaluations:\n  • avg_rouge_l: 0.566\n    💭 평균 ROUGE-L: 0.5664 (49개 항목)\n  • avg_conciseness: 0.490\n    💭 평균 간결성: 0.49 (49개 항목)\n\n🔗 Dataset Run:\n   https://cloud.langfuse.com/project/cmmki2iwm01esad07c65zx7l6/datasets/cmmpnrp5a012pad07tq46g1gp/runs/b1e9b332-7472-4d62-9826-2147b81b74f4


---

## **로컬 데이터를 사용한 실험**

데이터셋이 Langfuse에 업로드되지 않은 경우, 로컬 데이터로도 직접 실험을 실행할 수 있습니다.

In [26]:
# 로컬 테스트 데이터 정의
local_test_data = [
    {
        "input": "테슬라의 회장은 누구인가요?",
        "expected_output": "일론 머스크"
    },
    {
        "input": "테슬라는 언제 설립되었나요?",
        "expected_output": "2003년 7월"
    },
]

# 로컬 데이터용 Task 함수 (item이 딕셔너리)
def local_rag_task(*, item, **kwargs):
    """로컬 데이터를 위한 RAG 체인 실행"""
    question = item["input"]
    answer = openai_rag_chain.invoke(question)
    return answer

# 로컬 데이터에서 실험 실행
local_result = langfuse_client.run_experiment(
    name="Local_RAG_Test",
    description="로컬 데이터를 사용한 RAG 테스트",
    data=local_test_data,
    task=local_rag_task,
    evaluators=[rouge_evaluator, length_evaluator],
)

print(local_result.format())

Individual Results: Hidden (2 items)\n💡 Set include_item_results=True to view them\n\n──────────────────────────────────────────────────\n🧪 Experiment: Local_RAG_Test
📋 Run name: Local_RAG_Test - 2026-03-14T01:44:16.049946Z - 로컬 데이터를 사용한 RAG 테스트\n2 items\nEvaluations:\n  • rouge_l\n  • response_length\n\nAverage Scores:\n  • rouge_l: 0.200\n  • response_length: 47.000\n


---

## **[실습 문제]**

다음 요구사항에 맞춰 Experiment Runner SDK를 사용한 평가 시스템을 구현하세요.

### 요구사항

1. **Custom Evaluator 추가**
   - 키워드 일치 평가자: 정답에 포함된 핵심 키워드가 생성 답변에 포함되는지 확인
   - 응답 시간 평가자: RAG 체인 실행 시간 측정

2. **Run-level Evaluator 추가**
   - 전체 성능 요약: 모든 평가 지표의 평균을 계산하고 요약

3. **실험 실행**
   - 위에서 정의한 평가자들을 사용하여 실험 실행
   - Langfuse 대시보드에서 결과 확인

### 힌트

- `Evaluation` 클래스를 사용하여 평가 결과 반환
- Run-level evaluator는 `item_results` 파라미터를 받아 집계 수행
- `max_concurrency`를 조정하여 성능 최적화

In [27]:
# 여기에 코드를 작성하세요. 

---

## **LLM-as-Judge 평가 (OpenEvals 통합)**

Langfuse와 OpenEvals를 함께 사용하여 LLM 기반 평가를 수행할 수 있습니다.
(OpenEvals → Langfuse Evaluator 래핑)



In [28]:
# LLM-as-Judge Evaluator 구현 (OpenEvals 활용)

from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT, CONCISENESS_PROMPT
from langfuse import Evaluation

# 1. OpenEvals 평가자 생성
correctness_judge = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    feedback_key="correctness",
    model="openai:gpt-4.1-mini",
)

# 2. Langfuse Evaluator로 래핑
def correctness_llm_evaluator(*, input, output, expected_output, **kwargs):
    """LLM-as-Judge 정확성 평가"""
    try:
        result = correctness_judge(
            inputs=str(input),
            outputs=str(output),
            reference_outputs=str(expected_output),
        )
        
        # score가 boolean인 경우 float로 변환
        score = result.get('score', 0)
        if isinstance(score, bool):
            score = 1.0 if score else 0.0
        
        return Evaluation(
            name="llm_correctness",
            value=float(score),
            comment=result.get('comment', '')[:200]  # 코멘트 길이 제한
        )
    except Exception as e:
        return Evaluation(name="llm_correctness", value=0.0, comment=str(e))


# 3. 커스텀 LLM-as-Judge (한국어)
KOREAN_QUALITY_PROMPT = """당신은 RAG 시스템의 답변 품질을 평가하는 전문가입니다.

<질문>
{inputs}
</질문>

<생성된 답변>
{outputs}
</생성된 답변>

<참조 답변>
{reference_outputs}
</참조 답변>

다음 기준으로 0~1 점수를 부여하세요:
- 1.0: 정확하고 완전한 답변
- 0.7: 대부분 정확하지만 일부 누락
- 0.5: 부분적으로 정확
- 0.3: 관련은 있으나 대부분 부정확
- 0.0: 완전히 잘못됨
"""

korean_quality_judge = create_llm_as_judge(
    prompt=KOREAN_QUALITY_PROMPT,
    continuous=True,  # 0~1 연속 점수
    feedback_key="korean_quality",
    model="openai:gpt-4.1-mini",
)


def korean_quality_evaluator(*, input, output, expected_output, **kwargs):
    """한국어 품질 평가 (연속 점수)"""
    try:
        result = korean_quality_judge(
            inputs=str(input),
            outputs=str(output),
            reference_outputs=str(expected_output),
        )
        return Evaluation(
            name="korean_quality",
            value=float(result.get('score', 0)),
            comment=result.get('comment', '')[:200]
        )
    except Exception as e:
        return Evaluation(name="korean_quality", value=0.0, comment=str(e))


print("LLM-as-Judge Evaluator 정의 완료")
print("- correctness_llm_evaluator: 정확성 평가 (이진)")
print("- korean_quality_evaluator: 한국어 품질 평가 (연속 점수)")


LLM-as-Judge Evaluator 정의 완료
- correctness_llm_evaluator: 정확성 평가 (이진)
- korean_quality_evaluator: 한국어 품질 평가 (연속 점수)


In [29]:
import asyncio
import time
from langchain_openai import ChatOpenAI
from langfuse import Evaluation

# ============================================
# 비동기 RAG 체인 생성 (ainvoke 지원)
# ============================================

# LangChain의 ainvoke()를 활용한 진짜 비동기 Task
async def async_rag_task(*, item, **kwargs):
    """비동기 RAG 체인 실행 - LangChain ainvoke() 활용"""
    question = item.input if hasattr(item, 'input') else item["input"]
    # LangChain Runnable은 ainvoke()로 비동기 실행 가능
    answer = await openai_rag_chain.ainvoke(question)
    return answer

# 비동기 Evaluator 정의
async def async_rouge_evaluator(*, input, output, expected_output, **kwargs):
    """비동기 ROUGE 평가"""
    rouge_results = rouge_scorer_instance.score(
        str(expected_output), 
        str(output)
    )
    score = rouge_results['rougeL'].fmeasure
    return Evaluation(
        name="rouge_l",
        value=score,
        comment=f"ROUGE-L F1: {score:.4f}"
    )

async def async_length_evaluator(*, input, output, **kwargs):
    """비동기 응답 길이 평가"""
    length = len(str(output))
    return Evaluation(
        name="response_length",
        value=length,
        comment=f"응답 길이: {length}자"
    )

print("비동기 함수 정의 완료")
print("- async_rag_task: LangChain ainvoke()를 사용한 비동기 RAG Task")
print("- async_rouge_evaluator: 비동기 ROUGE 평가")
print("- async_length_evaluator: 비동기 응답 길이 평가")

비동기 함수 정의 완료
- async_rag_task: LangChain ainvoke()를 사용한 비동기 RAG Task
- async_rouge_evaluator: 비동기 ROUGE 평가
- async_length_evaluator: 비동기 응답 길이 평가


---

## **비동기 실험 실행 (Async)**

대용량 데이터셋의 경우 비동기 실행이 더 효율적입니다.

#### 비동기 Task 함수

```python
async def async_rag_task(*, item, **kwargs):
    question = item.input
    # 비동기 RAG 체인 실행
    answer = await async_rag_chain.ainvoke(question)
    return answer
```

#### 비동기 Evaluator

```python
async def async_evaluator(*, input, output, expected_output, **kwargs):
    # 비동기 평가 로직
    score = await async_calculate_score(output, expected_output)
    return Evaluation(name="async_metric", value=score)
```

#### 주요 이점

| 이점 | 설명 |
|------|------|
| **처리량 향상** | I/O 대기 시간 동안 다른 작업 수행 |
| **자원 효율성** | 스레드 풀보다 적은 메모리 사용 |
| **확장성** | 대용량 데이터셋에서 성능 향상 |

#### 주의사항

- `max_concurrency` 설정으로 API 레이트 리밋 관리
- LLM API 호출이 많은 경우 적절한 동시성 설정 필요


In [30]:
# ============================================
# 비동기 실험 실행 - 동기 vs 비동기 시간 비교
# ============================================
import time 

# 로컬 테스트 데이터 (소규모로 빠르게 비교)
async_test_data = [
    {"input": "테슬라의 회장은 누구인가요?", "expected_output": "일론 머스크"},
    {"input": "테슬라는 언제 설립되었나요?", "expected_output": "2003년 7월"},
    {"input": "테슬라의 첫 번째 차량은 무엇인가요?", "expected_output": "Roadster"},
    {"input": "테슬라의 에너지 제품에는 무엇이 있나요?", "expected_output": "Powerwall, Megapack, Solar Roof"},
    {"input": "Model 3는 언제 공개되었나요?", "expected_output": "2016년 4월"},
]

langfuse_client = get_client()

# --- 1) 동기 실험 ---
start_sync = time.time()

sync_result = langfuse_client.run_experiment(
    name="Sync_RAG_Benchmark",
    description="동기 실행 벤치마크",
    data=async_test_data,
    task=local_rag_task,
    evaluators=[rouge_evaluator, length_evaluator],
    max_concurrency=1,  # 순차 실행
)

sync_elapsed = time.time() - start_sync

# --- 2) 비동기 실험 ---
# 비동기 로컬 Task (item이 딕셔너리)
async def async_local_rag_task(*, item, **kwargs):
    question = item["input"]
    return await openai_rag_chain.ainvoke(question)

start_async = time.time()

async_result = langfuse_client.run_experiment(
    name="Async_RAG_Benchmark",
    description="비동기 실행 벤치마크",
    data=async_test_data,
    task=async_local_rag_task,  # async 함수 전달
    evaluators=[async_rouge_evaluator, async_length_evaluator],
    max_concurrency=5,  # 동시 5개 실행
)

async_elapsed = time.time() - start_async

# --- 결과 비교 ---
print("=" * 60)
print(f"  동기 실행 (max_concurrency=1): {sync_elapsed:.2f}초")
print(f"  비동기 실행 (max_concurrency=5): {async_elapsed:.2f}초")
speedup = sync_elapsed / async_elapsed if async_elapsed > 0 else 0
print(f"  속도 향상: {speedup:.1f}x")
print("=" * 60)
print()
print(async_result.format())

  동기 실행 (max_concurrency=1): 6.24초
  비동기 실행 (max_concurrency=5): 2.31초
  속도 향상: 2.7x

Individual Results: Hidden (5 items)\n💡 Set include_item_results=True to view them\n\n──────────────────────────────────────────────────\n🧪 Experiment: Async_RAG_Benchmark
📋 Run name: Async_RAG_Benchmark - 2026-03-14T01:44:25.969028Z - 비동기 실행 벤치마크\n5 items\nEvaluations:\n  • rouge_l\n  • response_length\n\nAverage Scores:\n  • rouge_l: 0.240\n  • response_length: 52.200\n
